In [1]:
import os
import numpy.random as rd
import matplotlib.pyplot as plt
from functools import partial
import numpy as np
import time
import pyvisa as pvisa
import qcodes as qc
from qcodes.instrument_drivers.yehlab_driver.QD import QD
import qcodes.instrument_drivers.nplab_drivers as npd
import qcodes.instrument_drivers.yehlab_driver as yd
from qcodes.instrument_drivers.stanford_research.SR830 import SR830
from qcodes.instrument_drivers.Keithley.Keithley_2636B import *

In [2]:
rm = pvisa.ResourceManager()
resources = rm.list_resources()
print("List")
for res in resources:
    print(res)

List
GPIB0::13::INSTR
GPIB0::2::INSTR
GPIB1::3::INSTR
GPIB1::8::INSTR
PXI0::4-0.0::INSTR
USB0::0x05E6::0x2636::4353546::0::INSTR
USB0::0x05E6::0x2636::4437673::0::INSTR
USB0::0x0957::0x3D18::MY59030016::0::INSTR
USB0::0x1313::0x8078::P0005244::0::INSTR
USB0::0x1313::0x8078::PM001388::0::INSTR
USB0::0x2EC7::0x6522::802557043747210033::0::INSTR
ASRL3::INSTR
ASRL5::INSTR


In [3]:
k2636B = Keithley2600('k2636B','USB0::0x05E6::0x2636::4437673::0::INSTR')

Connected to: Keithley Instruments Inc. 2636B (serial:4437673, firmware:3.3.5) in 0.16s


In [4]:
sr830=SR830('lockin1','GPIB0::3::INSTR')

[lockin1(SR830)] Could not connect at GPIB0::3::INSTR
Traceback (most recent call last):
  File "C:\Users\HP\.conda\envs\LL_Group_qcodes\lib\site-packages\qcodes\instrument\visa.py", line 133, in _connect_and_handle_error
    visa_handle, visabackend = self._open_resource(address, visalib)
  File "C:\Users\HP\.conda\envs\LL_Group_qcodes\lib\site-packages\qcodes\instrument\visa.py", line 160, in _open_resource
    resource = resource_manager.open_resource(address)
  File "C:\Users\HP\.conda\envs\LL_Group_qcodes\lib\site-packages\pyvisa\highlevel.py", line 3284, in open_resource
    res.open(access_mode, open_timeout)
  File "C:\Users\HP\.conda\envs\LL_Group_qcodes\lib\site-packages\pyvisa\resources\resource.py", line 282, in open
    self.session, status = self._resource_manager.open_bare_resource(
  File "C:\Users\HP\.conda\envs\LL_Group_qcodes\lib\site-packages\pyvisa\highlevel.py", line 3209, in open_bare_resource
    return self.visalib.open(self.session, resource_name, access_mode,

VisaIOError: VI_ERROR_RSRC_NFOUND (-1073807343): Insufficient location information or the requested device or resource is not present in the system.

In [4]:
k2636B.smua.mode()

'voltage'

In [5]:
k2636B.smub.mode()

'voltage'

In [6]:
#Define the input values (Vd)
vol_val = np.arange(-3, 3.01, 0.2)
vol_val[15]=0
print(np.shape(vol_val))
print(vol_val)

(31,)
[-3.  -2.8 -2.6 -2.4 -2.2 -2.  -1.8 -1.6 -1.4 -1.2 -1.  -0.8 -0.6 -0.4
 -0.2  0.   0.2  0.4  0.6  0.8  1.   1.2  1.4  1.6  1.8  2.   2.2  2.4
  2.6  2.8  3. ]


In [7]:
vol_val = np.arange(-3, 3.01, 0.2)
vol_val[15] = 0
x = np.arange(3, -3.01, -0.2)
x[15] = 0
vol_val = np.append(vol_val, x)
print(vol_val)
"""
print(x)


y = -1*vol_val
vol_val = np.append(vol_val, y)
print(np.shape(vol_val))
print(vol_val)
re_vol_val = vol_val* -1
print(re_vol_val)
"""

[-3.  -2.8 -2.6 -2.4 -2.2 -2.  -1.8 -1.6 -1.4 -1.2 -1.  -0.8 -0.6 -0.4
 -0.2  0.   0.2  0.4  0.6  0.8  1.   1.2  1.4  1.6  1.8  2.   2.2  2.4
  2.6  2.8  3.   3.   2.8  2.6  2.4  2.2  2.   1.8  1.6  1.4  1.2  1.
  0.8  0.6  0.4  0.2  0.  -0.2 -0.4 -0.6 -0.8 -1.  -1.2 -1.4 -1.6 -1.8
 -2.  -2.2 -2.4 -2.6 -2.8 -3. ]


'\nprint(x)\n\n\ny = -1*vol_val\nvol_val = np.append(vol_val, y)\nprint(np.shape(vol_val))\nprint(vol_val)\nre_vol_val = vol_val* -1\nprint(re_vol_val)\n'

In [17]:
# 設置電壓源和測量
k2636B.smua.limiti(10e-6)  # 設置電流上限
k2636B.smua.measure_autorange_i_enabled(1)  # 啟用自動範圍
k2636B.smua.sourcerange_v(20)  # 設置電壓源範圍

# 設置閘極電壓源和測量
k2636B.smub.limiti(10e-6)  # 設置電流上限
k2636B.smub.measure_autorange_i_enabled(1)  # 啟用自動範圍
k2636B.smub.sourcerange_v(20)  # 設置電壓源範圍


In [24]:
k2636B.smua.volt(0)
k2636B.smub.volt(10)
#k2636B.smua.output(0)
#k2636B.smub.output(0)




In [25]:
k2636B.smua.output(1)
k2636B.smub.output(1)
time.sleep(1)
data1, plot1=npd.single_param_sweep(k2636B.smua.volt, vol_val, 0.002, k2636B.smua.curr, k2636B.smua.volt, k2636B.smub.volt, k2636B.smub.curr, DataName='test') #,sr830.R, sr830.P, sr830.X, DataName='dark')
time.sleep(1)
k2636B.smua.volt(0)
k2636B.smub.volt(0)
k2636B.smua.output(0)
k2636B.smub.output(0)

Started at 2025-08-29 12:06:04
Received SIGINT, Will interrupt at first suitable time. Send second SIGINT to interrupt immediately.
DataSet:
   location = 'data/2025-08-29/#009_test_12-06-02'
   <Type>   | <array_id>           | <array.name> | <array.shape>
   Setpoint | k2636B_smua_volt_set | volt         | (62,)
   Measured | k2636B_smua_curr     | curr         | (62,)
   Measured | k2636B_smua_volt     | volt         | (62,)
   Measured | k2636B_smub_volt     | volt         | (62,)
   Measured | k2636B_smub_curr     | curr         | (62,)
Finished at 2025-08-29 12:06:17
Keyboard Interrupt


In [117]:
x = 20
while x >= -10:
    if x == -10:
        time.sleep(1)
        k2636B.smua.volt(0)
        k2636B.smub.volt(0)
        k2636B.smua.output(0)
        k2636B.smub.output(0)
    else:
        time.sleep(1)
        k2636B.smua.volt(0)
        k2636B.smub.volt(0)
        k2636B.smua.output(0)
        k2636B.smub.output(0)
        time.sleep(1)
        k2636B.smub.volt(x)
        time.sleep(3)
        k2636B.smua.output(1)
        k2636B.smub.output(1)
        time.sleep(3)  # 稳定
        data1 , plot1=npd.single_param_sweep(k2636B.smua.volt, vol_val, 0.2, k2636B.smua.curr, k2636B.smua.volt, k2636B.smub.volt, sr830.R, sr830.P, sr830.X, DataName='Dark'+str(x))
        time.sleep(1)
    x=x-10

Started at 2025-08-19 23:10:35
DataSet:
   location = 'data/2025-08-19/#060_LCP-225-OAM220_23-10-31'
   <Type>   | <array_id>           | <array.name> | <array.shape>
   Setpoint | k2636B_smua_volt_set | volt         | (122,)
   Measured | k2636B_smua_curr     | curr         | (122,)
   Measured | k2636B_smua_volt     | volt         | (122,)
   Measured | k2636B_smub_volt     | volt         | (122,)
   Measured | lockin1_R            | R            | (122,)
   Measured | lockin1_P            | P            | (122,)
   Measured | lockin1_X            | X            | (122,)
Finished at 2025-08-19 23:11:52
Started at 2025-08-19 23:12:05
DataSet:
   location = 'data/2025-08-19/#061_LCP-225-OAM210_23-12-01'
   <Type>   | <array_id>           | <array.name> | <array.shape>
   Setpoint | k2636B_smua_volt_set | volt         | (122,)
   Measured | k2636B_smua_curr     | curr         | (122,)
   Measured | k2636B_smua_volt     | volt         | (122,)
   Measured | k2636B_smub_volt     | volt   

In [20]:
time_val = np.arange(0,601, 1)
print(np.shape(time_val))
print(time_val)

(601,)
[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197
 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215
 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233
 234 235 236 237 238 239 240 241 242 243 244

In [21]:
k2636B.smua.volt(-3)
k2636B.smub.volt(20)

In [83]:
k2636B.smua.output(1)
k2636B.smub.output(1)
time.sleep(1)
data1 , plot1=npd.single_param_sweep(sr830.Time, time_val, 1, k2636B.smua.curr, sr830.R, sr830.P, DataName='PhotoresponseTest')
time.sleep(1)
k2636B.smua.volt(0)
k2636B.smub.volt(0)
k2636B.smua.output(0)
k2636B.smub.output(0)

Started at 2025-08-19 21:40:55
DataSet:
   location = 'data/2025-08-19/#022_PhotoresponseTest_21-40-53'
   <Type>   | <array_id>       | <array.name> | <array.shape>
   Setpoint | lockin1_Time_set | Time         | (601,)
   Measured | k2636B_smua_curr | curr         | (601,)
   Measured | lockin1_R        | R            | (601,)
   Measured | lockin1_P        | P            | (601,)
Finished at 2025-08-19 21:40:58
Keyboard Interrupt


In [98]:
k2636B.smua.volt(-3)
k2636B.smub.volt(20)

In [99]:
k2636B.smua.output(1)
k2636B.smub.output(1)

In [100]:
sr830.write('AGAN')

In [101]:
k2636B.smua.volt(0)
k2636B.smub.volt(0)
k2636B.smua.output(0)
k2636B.smub.output(0)